# Imports

In [1]:
from dotenv import load_dotenv
import json
from anthropic import Anthropic
from datetime import datetime
from anthropic.types import ToolParam
load_dotenv()

True

# Create the Anthropic Client

In [2]:
client = Anthropic()
model = "claude-haiku-4-5"

# Tool Function Call

In [7]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [8]:
get_current_datetime()

'2026-06-29 14:18:04'

# Tool Schema

In [6]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

# Client Call without Tools

In [3]:
messages = [{
    "role": "user",
    "content": "What is the current date and time?"
}]

response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages
)

print("Response:\n", response)
print("\n")
print("Stop reason:\n", response.stop_reason)

Response:
 Message(id='msg_01E2x9aPSe4quoaRKNu3icAY', container=None, content=[TextBlock(citations=None, text='I don\'t have access to real-time information, so I can\'t tell you the current date and time. \n\nTo find this, you can:\n- Check your device (phone, computer, etc.)\n- Search "current time" online\n- Ask a voice assistant like Siri, Alexa, or Google Assistant\n\nIs there something time-related I can help you with?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=86, server_tool_use=None, service_tier='standard'))


Stop reason:
 end_turn


# Client Call with Tool

In [4]:
messages = [{
    "role": "user",
    "content": \
    "What is the current date and time?"
}]


In [9]:
response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema],
)

In [10]:
print("Response:\n", response)

Response:
 Message(id='msg_01GM3o67DBUX3i7xy93NTptd', container=None, content=[ToolUseBlock(id='toolu_01VrddKY4MHuzLTTcStyAHyk', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=612, output_tokens=39, server_tool_use=None, service_tier='standard'))


In [11]:
messages.append({
    "role": "assistant",
    "content": response.content
})

In [12]:
tool_result = get_current_datetime()

In [13]:
print(tool_result)

2026-06-29 14:18:54


In [14]:
tool_request = response.content[0]
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_result),
            "is_error": False,
        }
    ]
})

In [15]:
print(messages)

[{'role': 'user', 'content': 'What is the current date and time?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01VrddKY4MHuzLTTcStyAHyk', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01VrddKY4MHuzLTTcStyAHyk', 'content': '"2026-06-29 14:18:54"', 'is_error': False}]}]


In [16]:
response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema]
)

In [17]:
print(response)

Message(id='msg_01V2yRvYqoPA1UNq22sqWEBT', container=None, content=[TextBlock(citations=None, text='The current date and time is **June 29, 2026 at 2:18:54 PM** (14:18:54 in 24-hour format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=676, output_tokens=42, server_tool_use=None, service_tier='standard'))
